# 2995. Viewers Turned Streamers

```text
# 2995. Viewers Turned Streamers
# Description
# Table: Sessions
# +---------------+----------+
# | Column Name   | Type     |
# +---------------+----------+
# | user_id       | int      |
# | session_start | datetime |
# | session_end   | datetime |
# | session_id    | int      |
# | session_type  | enum     |
# +---------------+----------+
# session_id is column of unique values for this table.
# session_type is an ENUM (category) type of (Viewer, Streamer).
# This table contains user id, session start, session end, session id and session type.
# Write a solution to find the number of streaming sessions for users whose first session
# was as a viewer.
# Return the result table ordered by count of streaming sessions, user_id in descending
# order.
# Input: 
# Sessions table:
# +---------+---------------------+---------------------+------------+--------------+
# | user_id | session_start       | session_end         | session_id | session_type | 
# +---------+---------------------+---------------------+------------+--------------+
# | 101     | 2023-11-06 13:53:42 | 2023-11-06 14:05:42 | 375        | Viewer       |  
# | 101     | 2023-11-22 16:45:21 | 2023-11-22 20:39:21 | 594        | Streamer     |   
# | 102     | 2023-11-16 13:23:09 | 2023-11-16 16:10:09 | 777        | Streamer     | 
# | 102     | 2023-11-17 13:23:09 | 2023-11-17 16:10:09 | 778        | Streamer     | 
# | 101     | 2023-11-20 07:16:06 | 2023-11-20 08:33:06 | 315        | Streamer     | 
# | 104     | 2023-11-27 03:10:49 | 2023-11-27 03:30:49 | 797        | Viewer       | 
# | 103     | 2023-11-27 03:10:49 | 2023-11-27 03:30:49 | 798        | Streamer     |  
# +---------+---------------------+---------------------+------------+--------------+
# Output: 
# +---------+----------------+
# | user_id | sessions_count | 
# +---------+----------------+
# | 101     | 2              | 
# +---------+----------------+
# Explanation
# - user_id 101, initiated their initial session as a viewer on 2023-11-06 at 13:53:42,
# followed by two subsequent sessions as a Streamer, the count will be 2.
# - user_id 102, although there are two sessions, the initial session was as a Streamer,
# so this user will be excluded.
# - user_id 103 participated in only one session, which was as a Streamer, hence, 
# it won't be considered.
# - User_id 104 commenced their first session as a viewer but didn't have any 
# subsequent sessions, therefore, they won't be included in the final count. 
# Output table is ordered by sessions count and user_id in descending order.
```

In [ ]:
import pandas as pd

sessions = pd.DataFrame({
    "user_id": [101, 101, 102, 102, 101, 104, 103],
    "session_start": [
        "2023-11-06 13:53:42",
        "2023-11-22 16:45:21",
        "2023-11-16 13:23:09",
        "2023-11-17 13:23:09",
        "2023-11-20 07:16:06",
        "2023-11-27 03:10:49",
        "2023-11-27 03:10:49"
    ],
    "session_end": [
        "2023-11-06 14:05:42",
        "2023-11-22 20:39:21",
        "2023-11-16 16:10:09",
        "2023-11-17 16:10:09",
        "2023-11-20 08:33:06",
        "2023-11-27 03:30:49",
        "2023-11-27 03:30:49"
    ],
    "session_id": [375, 594, 777, 778, 315, 797, 798],
    "session_type": ["Viewer", "Streamer", "Streamer", "Streamer", "Streamer", "Viewer", "Streamer"]
})


sessions["session_start"] = pd.to_datetime(sessions["session_start"])
sessions["session_end"] = pd.to_datetime(sessions["session_end"])


df = (
    sessions
    .sort_values(by = ['user_id','session_start'])
    .assign(
        user_rank = lambda d: d.groupby('user_id')['session_start']
                                .cumcount()+1,
        # sessions_count = lambda d: d.query("session_type == 'Streamer' ")
        #                              .groupby('user_id')['user_id']
        #                              .transform('count')
        #                              .fillna(0) # having na problem
        sessions_count = lambda d: (
                (d['session_type'] == 'Streamer')
                .groupby(d['user_id'])
                .transform('sum')
        )
    )
    .loc[lambda d: (d['sessions_count'] >0) & 
                   (d['session_type'] == 'Viewer') & 
                   (d['user_rank'] == 1)
        ,
        ['user_id','sessions_count']
        ]
    .sort_values(by = ['sessions_count','user_id'], ascending= [False, False])
)

print(df)